# hw1


Choose **one** of the three models below (your choice, or assigned by the instructors). All are sequential — no residual connections — and are built only from convolution, BatchNorm, ReLU, pooling and linear layers. All run in `eval()` mode, FP32. Every convolution uses `padding = k // 2` and `bias=False`; every ReLU is `inplace=True`; input is `3 × S × S`; 100 output classes.

**Model**
| Layers | Output resolution |
|--------|-------------------|
| Conv7×7 s2 3→32, MaxPool 3×3 s2 p1 | S/4 |
| Conv5×5 32→64 | S/4 |
| Conv3×3 s2 64→128 | S/8 |
| Conv1×1 128→256 | S/8 |
| Conv3×3 s2 256→256 | S/16 |
| Conv1×1 256→512 | S/16 |
| head: GlobalAvgPool, Linear 512→256, ReLU, Linear 256→100 | — |



## MODEL

In [1]:
# models.py

import torch
import torch.nn as nn

class SimpleModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3, bias=False)
    self.pooling = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2, bias=False)
    self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False)
    self.conv4 = nn.Conv2d(128, 256, kernel_size=1, stride=1, padding=0, bias=False)
    self.conv5 = nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=False)
    self.conv6 = nn.Conv2d(256, 512, kernel_size=1, stride=1, padding=0, bias=False)

    self.gap = nn.AdaptiveAvgPool2d(1)
    self.linear1 = nn.Linear(512, 256)
    self.linear2 = nn.Linear(256, 100)
    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
      x = self.conv1(x)
      x = self.relu(x)
      x = self.pooling(x)

      x = self.conv2(x)
      x = self.relu(x)

      x = self.conv3(x)
      x = self.relu(x)

      x = self.conv4(x)
      x = self.relu(x)

      x = self.conv5(x)
      x = self.relu(x)

      x = self.conv6(x)
      x = self.relu(x)

      x = self.gap(x)
      x = torch.flatten(x, 1)

      x = self.linear1(x)
      x = self.relu(x)

      x = self.linear2(x)
      return x

## Equations.py

In [7]:
# equations.py

import numpy as np


N_PARAMS = 1040324
FP32_BYTES = 4

def layer_costs(image_size, batch):
  S = np.asarray(image_size, dtype=np.float64)
  B = np.asarray(batch, dtype=np.float64)

  return [
    # name, FLOPs, bytes
    ("conv1", 2352 * B * S**2, 4 * (11 * B * S**2 + 4704)),
    ("relu1", 0, 4 * 16 * B * S**2),
    ("pool", 0, 4 * 10 * B * S**2),
    ("conv2", 6400 * B * S**2, 4 * (6 * B * S**2 + 51200)),
    ("relu2", 0,4 * 8 * B * S**2),
    ("conv3", 2304 * B * S**2, 4 * (6 * B * S**2 + 73728)),
    ("relu3", 0, 4 * 4 * B * S**2),
    ("conv4", 1024 * B * S**2, 4 * (6 * B * S**2 + 32768)),
    ("relu4", 0, 4 * 8 * B * S**2),
    ("conv5", 4608 * B * S**2, 4 * (5 * B * S**2 + 589824)),
    ("relu5", 0, 4 * 2 * B * S**2),
    ("conv6", 1024 * B * S**2, 4 * (3 * B * S**2 + 131072)),
    ("relu6", 0, 4 * 4 * B * S**2),
    ("gap", 0, 4 * (2 * B * S**2 + 512 * B)),
    ("linear1", 2 * 512 * 256 * B, 4 * (768 * B + 131328)),
    ("relu_head", 0, 4 * 512 * B),
    ("linear2", 2 * 256 * 100 * B, 4 * (356 * B + 25700)),
  ]

def flops(image_size, batch):
  S = np.asarray(image_size, dtype=np.float64)
  B = np.asarray(batch, dtype=np.float64)

  return 17712 * B * S**2 + 313344*B


def memory(image_size, batch):
  S = np.asarray(image_size, dtype=np.float64)
  B = np.asarray(batch, dtype=np.float64)

  # 1040324 = count params
  # 4 = fp32
  return 4 * (1040324 + 11 * B * S**2)


def bytes_moved(image_size, batch):
  S = np.asarray(image_size, dtype=np.float64)
  B = np.asarray(batch, dtype=np.float64)

  # 1040324 = count params
  # 4 = fp32
  return 4 * (91 * B * S**2 + 2148 * B + 1040324)

def latency(image_size, batch, theta):
  t_launch = theta["t_launch"]
  BW = theta["BW"]
  P = theta["P"]

  result = 0.0

  for _, f, nbytes in layer_costs(image_size, batch):
    result = result + np.maximum(
      t_launch,
      np.maximum(
        nbytes / BW,
        f / P,
      )
    )

  return result

def energy(image_size, batch, theta_energy):
    T = latency(
        image_size,
        batch,
        theta_energy["latency_theta"],
    )

    return (
        theta_energy["p_0"] * T
        + theta_energy["energy_per_byte"] * bytes_moved(image_size, batch)
        + theta_energy["energy_per_flop"] * flops(image_size, batch)
    )

In [3]:
!pip install nvidia-ml-py

## Measure.py

In [4]:
# measure.py
import gc
import math
import time

import pandas as pd
import numpy as np
from pathlib import Path
import torch
import pynvml

#from models import SimpleModel
# from equations import flops, memory, bytes_moved

BASE_IMAGE_SIZES = [32, 64, 128, 224, 256, 384, 512, 1024]
BASE_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128, 256]

EXTRA_IMAGE_SIZES = [48, 192, 304, 352]
EXTRA_BATCH_SIZES = [7, 33, 67]

IMAGE_SIZES = sorted(BASE_IMAGE_SIZES + EXTRA_IMAGE_SIZES)
BATCH_SIZES = sorted(BASE_BATCH_SIZES + EXTRA_BATCH_SIZES)

N_WARMAP = 10
N_LATENCY_RUNS = 50
N_ENERGY_RUNS = 100

OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "measurements.csv"

torch.backends.cudnn.benchmark = False        # main results: PyTorch's default heuristics choose the kernel
torch.backends.cudnn.allow_tf32 = False       # so that FP32 means FP32 on Ampere and newer GPUs
torch.backends.cuda.matmul.allow_tf32 = False

device = torch.device("cuda")

def warmup(model,x):
  with torch.inference_mode():
    for _ in range(N_WARMAP):
      y = model(x)

  torch.cuda.synchronize()

  del y

def measure_latency(model, x):
  times = []

  with torch.inference_mode():
    for _ in range(N_LATENCY_RUNS):
      start = torch.cuda.Event(enable_timing=True)
      end = torch.cuda.Event(enable_timing=True)

      start.record()
      y = model(x)
      end.record()

      end.synchronize()

      # time in seconds
      elapsed_time = start.elapsed_time(end) / 1000
      times.append(elapsed_time)

      del y

  # median
  latency = np.median(times)

  return float(latency)

def measure_memory(model, x):
  torch.cuda.synchronize()

  torch.cuda.reset_peak_memory_stats()

  with torch.inference_mode():
    y = model(x)

  torch.cuda.synchronize()
  memory = int(torch.cuda.max_memory_allocated())

  del y

  return memory

# energy measurement
def init_nvml():
  pynvml.nvmlInit()
  handle = pynvml.nvmlDeviceGetHandleByIndex(0)
  return handle

def read_total_energy_joules(handle):
  energy = pynvml.nvmlDeviceGetTotalEnergyConsumption(handle) / 1000
  return energy


def measure_energy(model, x, handle, n_runs):
  torch.cuda.synchronize()

  energy_before = read_total_energy_joules(handle)

  with torch.inference_mode():
    for _ in range(n_runs):
      y = model(x)

  torch.cuda.synchronize()

  energy_after = read_total_energy_joules(handle)

  del y

  return float((energy_after - energy_before) / n_runs)

def is_validation_point(image_size, batch_size):
  return (image_size in EXTRA_IMAGE_SIZES) or (batch_size in EXTRA_BATCH_SIZES)

def measure_configuration(model, image_size, batch_size, nvml_handle):
  S = image_size
  B = batch_size
  x = None

  try:
    x = torch.randn(B, 3, S, S, device="cuda", dtype=torch.float32)

    warmup(model, x)

    latency = measure_latency(model, x)
    memory = measure_memory(model, x)
    n_energy_runs = max(10, int(np.ceil(0.5 / latency)))
    energy = measure_energy(model, x, nvml_handle, n_energy_runs)

    row = {
      "B": B,
      "S": S,
      "latency": latency,
      "memory": memory,
      "energy": energy,
      "is_validation": is_validation_point(S, B),
      "oom": False
    }
    return row

  except torch.cuda.OutOfMemoryError:
    print("OOM")
    if x is not None:
      del x
      x = None

    gc.collect()
    torch.cuda.empty_cache()

    row = {
      "B": B,
      "S": S,
      "latency": np.nan,
      "memory": np.nan,
      "energy": np.nan,
      "is_validation": is_validation_point(S, B),
      "oom": True
    }

    return row

  finally:
    if x is not None:
      del x
    gc.collect()

def main():

  if not torch.cuda.is_available():
    raise RuntimeError(
      "CUDA GPU is required for this homework."
    )


  torch.manual_seed(42)
  torch.cuda.manual_seed(42)

  model = SimpleModel().to(device)
  model = model.cuda().eval()

  nvml_handle = init_nvml()

  # ==== INFO
  print("GPU" + str(torch.cuda.get_device_name(0)))
  total_memory = torch.cuda.get_device_properties(0).total_memory

  print(
    "GPU memory:",
    round(total_memory / 1024**3, 2),
    "GiB",
  )

  print()

  print("PyTorch:", torch.__version__)
  print("CUDA:", torch.version.cuda)
  print("cuDNN:", torch.backends.cudnn.version())

  print()

  print("cudnn.benchmark =", torch.backends.cudnn.benchmark)
  print(
    "cudnn.allow_tf32 =",
    torch.backends.cudnn.allow_tf32,
  )
  print(
    "matmul.allow_tf32 =",
    torch.backends.cuda.matmul.allow_tf32,
  )

  print()

  # ==== RUNS
  rows = []

  total = len(IMAGE_SIZES) * len(BATCH_SIZES)
  current = 0
  for S in IMAGE_SIZES:
    for B in BATCH_SIZES:
      current += 1

      print(
        f"[{current:3d}/{total}] "
        f"S={S:3d}, B={B:3d}",
        end=" | ",
        flush=True,
      )

      row = measure_configuration(
        model=model,
        image_size=S,
        batch_size=B,
        nvml_handle=nvml_handle,
      )

      rows.append(row)

      if row["oom"]:
        print()

      else:
        print(
            f"latency = "
            f"{row['latency'] * 1000:.3f} ms | "
            f"memory = "
            f"{row['memory'] / 1024**2:.2f} MiB | "
            f"energy = "
            f"{row['energy']:.6f} J"
        )

  df = pd.DataFrame(rows)

  df.to_csv(
      OUT_PATH,
      index=False,
  )

  pynvml.nvmlShutdown()

  print()
  print("==============================")
  print("Measurement finished")
  print("==============================")

  print("Total configurations:", len(df))
  print("OOM:", int(df["oom"].sum()))
  print(
      "Validation:",
      int(df["is_validation"].sum()),
  )

  print()
  print("Saved to:")
  print(OUT_PATH)

if __name__ == "__main__":
  main()

GPUTesla T4
GPU memory: 14.56 GiB

PyTorch: 2.11.0+cu128
CUDA: 12.8
cuDNN: 91900

cudnn.benchmark = False
cudnn.allow_tf32 = False
matmul.allow_tf32 = False

[  1/144] S= 32, B=  1 | latency = 0.723 ms | memory = 17.89 MiB | energy = 0.014945 J
[  2/144] S= 32, B=  2 | latency = 0.481 ms | memory = 18.00 MiB | energy = 0.022158 J
[  3/144] S= 32, B=  4 | latency = 0.519 ms | memory = 18.21 MiB | energy = 0.020498 J
[  4/144] S= 32, B=  7 | latency = 0.681 ms | memory = 18.53 MiB | energy = 0.030709 J
[  5/144] S= 32, B=  8 | latency = 0.684 ms | memory = 18.63 MiB | energy = 0.032082 J
[  6/144] S= 32, B= 16 | latency = 0.678 ms | memory = 14.16 MiB | energy = 0.026724 J
[  7/144] S= 32, B= 32 | latency = 0.927 ms | memory = 15.22 MiB | energy = 0.033550 J
[  8/144] S= 32, B= 33 | latency = 0.612 ms | memory = 15.29 MiB | energy = 0.032965 J
[  9/144] S= 32, B= 64 | latency = 0.737 ms | memory = 17.34 MiB | energy = 0.051935 J
[ 10/144] S= 32, B= 67 | latency = 0.776 ms | memory = 17.5

## Calibrate.py


In [19]:
# calibrate.py
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares, lsq_linear

from equations import latency, energy, flops, memory, bytes_moved

OUT_DIR = Path("results")
THETA_PATH = OUT_DIR / "theta.json"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, np.float64)
    y_pred = np.asarray(y_pred, np.float64)
    err = y_true - y_pred
    rel = np.abs(err) / np.maximum(np.abs(y_true), 1e-12)
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mape_percent": float(np.mean(rel) * 100),
        "median_ape_percent": float(np.median(rel) * 100),
    }


def _evaluate(df, col, predict):
    if len(df) == 0:
        return None
    S = df["S"].to_numpy(np.float64)
    B = df["B"].to_numpy(np.float64)
    return regression_metrics(df[col].to_numpy(np.float64), predict(S, B))


def _both_splits(calib, val, col, predict):
    return {
        "calibration": _evaluate(calib, col, predict),
        "validation": _evaluate(val, col, predict),
    }


def fit_latency(df):
    S = df["S"].to_numpy(np.float64)
    B = df["B"].to_numpy(np.float64)
    measured = df["latency"].to_numpy(np.float64)

    params = {
        "t_launch": (5e-6, 1e-8, 1e-2),
        "BW":       (2e11, 1e8,  1e13),
        "P":        (5e12, 1e9,  1e15),
    }
    keys = tuple(params)

    def unpack(q):
        return dict(zip(keys, np.exp(q)))

    def residuals(q):
        return np.log(latency(S, B, unpack(q))) - np.log(measured)

    q0 = np.log([params[k][0] for k in keys])
    lo = np.log([params[k][1] for k in keys])
    hi = np.log([params[k][2] for k in keys])

    result = least_squares(residuals, x0=q0, bounds=(lo, hi))
    return unpack(result.x), result


def fit_energy(df, latency_theta):
    S = df["S"].to_numpy(np.float64)
    B = df["B"].to_numpy(np.float64)
    measured = df["energy"].to_numpy(np.float64)

    X = np.column_stack([
        latency(S, B, latency_theta),
        bytes_moved(S, B) / 1e9,
        flops(S, B) / 1e9,
    ])
    result = lsq_linear(X, measured, bounds=(0, np.inf))

    theta = {
        "p_0": float(result.x[0]),
        "energy_per_byte": float(result.x[1] / 1e9),
        "energy_per_flop": float(result.x[2] / 1e9),
        "latency_theta": {k: float(v) for k, v in latency_theta.items()},
    }
    return theta, result

def plot_surface_and_curves(df_calib, df_val, column, predict, zlabel,
                            filename, scale=1.0, title=""):
    df_all = pd.concat([df_calib, df_val])

    S_grid, B_grid = np.meshgrid(
        np.linspace(df_all["S"].min(), df_all["S"].max(), 40),
        np.linspace(df_all["B"].min(), df_all["B"].max(), 40),
    )
    Z = predict(S_grid, B_grid) * scale

    fig = plt.figure(figsize=(14, 11))
    fig.suptitle(title, fontsize=18)

    for i, (sub, split_name, marker) in enumerate(
        [(df_calib, "Calibration", "o"), (df_val, "Validation", "^")], 1
    ):
        ax = fig.add_subplot(2, 2, i, projection="3d")
        S = sub["S"].to_numpy(np.float64)
        B = sub["B"].to_numpy(np.float64)
        measured = sub[column].to_numpy(np.float64) * scale
        predicted = predict(S, B) * scale

        ax.plot_surface(S_grid, B_grid, Z, alpha=0.22, color="blue")
        ax.scatter(S, B, measured, color="tab:blue", marker=marker,
                   s=35, label="Measured")
        ax.scatter(S, B, predicted, color="tab:orange", marker="x",
                   s=45, label="Predicted")

        ax.set_xlabel("Image size S, px")
        ax.set_ylabel("Batch size B")
        ax.set_zlabel(zlabel)
        ax.set_title(split_name)
        ax.legend()

    for i, (sub, split_name) in enumerate(
        [(df_calib, "Calibration"), (df_val, "Validation")], 3
    ):
        ax = fig.add_subplot(2, 2, i)

        S = sub["S"].to_numpy(np.float64)
        B = sub["B"].to_numpy(np.float64)
        workload = B * S ** 2
        measured = sub[column].to_numpy(np.float64) * scale
        predicted = predict(S, B) * scale

        order = np.argsort(workload)
        workload, measured, predicted = (
            workload[order], measured[order], predicted[order]
        )

        m = regression_metrics(measured, predicted)

        ax.plot(workload, measured, "o-", color="tab:blue",
                markersize=4, label="Measured")
        ax.plot(workload, predicted, "x-", color="tab:orange",
                markersize=4, label="Predicted")

        ax.set_xscale("log")
        ax.set_xlabel(r"Workload $BS^2$")
        ax.set_ylabel(zlabel)
        ax.set_title(
            f"{split_name}: measured vs predicted\n"
            f"MAPE={m['mape_percent']:.1f}%, "
            f"MedianAPE={m['median_ape_percent']:.1f}%"
        )
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def plot_parity_and_errors(df_calib, df_val, column, predict, label,
                           filename, scale=1.0, log=False, title=""):
    fig, axes = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(title, fontsize=18)

    splits = [
        (df_calib, "Calibration", "o", "tab:blue"),
        (df_val,   "Validation",  "^", "tab:orange"),
    ]

    for col_id, (sub, split_name, marker, point_color) in enumerate(splits):
        S = sub["S"].to_numpy(np.float64)
        B = sub["B"].to_numpy(np.float64)
        measured = sub[column].to_numpy(np.float64) * scale
        predicted = predict(S, B) * scale
        workload = B * S ** 2

        abs_err = np.abs(predicted - measured)
        ape = abs_err / np.maximum(np.abs(measured), 1e-12) * 100
        m = regression_metrics(measured, predicted)

        # parity
        ax = axes[0, col_id]
        x, y = measured.copy(), predicted.copy()
        if log:
            mask = (x > 0) & (y > 0)
            x, y = x[mask], y[mask]

        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        ax.scatter(x, y, color=point_color, marker=marker,
                   alpha=0.75, label=split_name)
        ax.plot([lo, hi], [lo, hi], "k--", lw=1.5, label="Ideal")

        if log:
            ax.set_xscale("log")
            ax.set_yscale("log")

        ax.set_xlabel(f"Measured {label}")
        ax.set_ylabel(f"Predicted {label}")
        ax.set_title(f"{split_name}: predicted vs measured")
        ax.grid(alpha=0.3)
        ax.legend()

        # APE
        ax = axes[1, col_id]
        order = np.argsort(workload)
        ax.plot(workload[order], ape[order], "o-", color="tab:purple",
                markersize=4, label="APE")
        ax.axhline(m["mape_percent"], color="tab:red", ls="--",
                   label=f"MAPE = {m['mape_percent']:.1f}%")
        ax.axhline(m["median_ape_percent"], color="black", ls=":",
                   label=f"Median APE = {m['median_ape_percent']:.1f}%")
        ax.set_xscale("log")
        ax.set_xlabel(r"Workload $BS^2$")
        ax.set_ylabel("Absolute percentage error, %")
        ax.set_title(f"{split_name}: relative prediction error")
        ax.grid(alpha=0.3)
        ax.legend()

        # pointwise absolute error
        ax = axes[2, col_id]
        ax.plot(workload[order], abs_err[order], "o-", color="tab:green",
                markersize=4, label="Absolute error")
        ax.axhline(m["mae"], color="tab:red", ls="--",
                   label=f"MAE = {m['mae']:.3g}")
        ax.set_xscale("log")
        ax.set_xlabel(r"Workload $BS^2$")
        ax.set_ylabel(f"Absolute error, {label.split(',')[-1].strip()}")
        ax.set_title(f"{split_name}: absolute prediction error")
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(FIG_DIR / filename, dpi=200)
    plt.close()


def main():
    df = pd.read_csv(OUT_DIR / "measurements.csv")

    valid = df[
        (~df["oom"])
        & np.isfinite(df["latency"])
        & np.isfinite(df["energy"])
        & np.isfinite(df["memory"])
    ].copy()
    calib = valid[~valid["is_validation"]]
    val = valid[valid["is_validation"]]

    if len(calib) == 0:
        raise RuntimeError("No calibration points available.")

    latency_theta, _ = fit_latency(calib)
    energy_theta, _ = fit_energy(calib, latency_theta)

    mem_predict = lambda S, B: memory(S, B)
    lat_predict = lambda S, B: latency(S, B, latency_theta)
    en_predict = lambda S, B: energy(S, B, energy_theta)

    metrics = {
        "memory":  _both_splits(calib, val, "memory",  mem_predict),
        "latency": _both_splits(calib, val, "latency", lat_predict),
        "energy":  _both_splits(calib, val, "energy",  en_predict),
    }

    output = {
        "latency": {k: float(v) for k, v in latency_theta.items()},
        "energy": energy_theta,
        "metrics": metrics,
    }
    with open(THETA_PATH, "w") as f:
        json.dump(output, f, indent=4)

    plot_surface_and_curves(
        calib, val, "memory", mem_predict,
        "Peak memory, MiB", "memory_surface.png",
        scale=1 / 1024 ** 2,
        title="Memory: measured values vs analytical prediction",
    )
    plot_parity_and_errors(
        calib, val, "memory", mem_predict, "memory, MiB",
        "memory_parity.png", scale=1 / 1024 ** 2,
        title="Memory: prediction quality",
    )

    plot_surface_and_curves(
        calib, val, "latency", lat_predict,
        "Latency, ms", "latency_surface.png",
        scale=1000,
        title="Latency: measured values vs calibrated analytical prediction",
    )
    plot_parity_and_errors(
        calib, val, "latency", lat_predict, "latency, ms",
        "latency_parity.png", scale=1000, log=True,
        title="Latency: prediction quality",
    )

    plot_surface_and_curves(
        calib, val, "energy", en_predict,
        "Energy, J", "energy_surface.png",
        title="Energy: measured values vs calibrated analytical prediction",
    )
    plot_parity_and_errors(
        calib, val, "energy", en_predict, "energy, J",
        "energy_parity.png", log=True,
        title="Energy: prediction quality",
    )


if __name__ == "__main__":
    main()